# Persiapan Data

In [89]:
# ======================
# Setup
# ======================
from pathlib import Path
import pandas as pd
import numpy as np
import requests

# Lokasi data mentah
folder_data = Path(
    r"C:\Users\Acer\OneDrive\Pictures\Kuliah\Data Sekunder"
)

# Pastikan folder ditemukan
print("Folder ditemukan:", folder_data.exists())

Folder ditemukan: True


In [90]:
# ======================
# Cek File Sumber
# ======================

file_sumber = [
    "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2020.xlsx",
    "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2021.xlsx",
    "banyaknya-kecelakaan-lalu-lintas.xls",
    "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2020.csv",
    "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2021.csv",
    "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2020.csv",
    "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2021.csv"
]

for nama_file in file_sumber:
    if (folder_data / nama_file).is_file():
        print("OK:", nama_file)
    else:
        print("TIDAK DITEMUKAN:", nama_file)

OK: Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2020.xlsx
OK: Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2021.xlsx
OK: banyaknya-kecelakaan-lalu-lintas.xls
OK: Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2020.csv
OK: Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2021.csv
OK: Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2020.csv
OK: Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2021.csv


In [91]:
# Baca file 2020
df_pesawat_2020 = pd.read_excel(
    folder_data / "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2020.xlsx"
)

# Baca file 2021
df_pesawat_2021 = pd.read_excel(
    folder_data / "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2021.xlsx"
)

df_pesawat_2020

,Jumlah Debarkasi Penumpang,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Debarkasi Penumpang Penerbangan Domesti...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,Nopember,Desember,Tahunan
3,Jumlah Penumpang,174381,204459,139860,30620,1657,16147,38496,43168,48207,59601,71838,76909,905343


In [92]:
# ======================
# Cleaning Data Penerbangan Domestik
# ======================

daftar_bulan = [
    "Januari", "Februari", "Maret", "April",
    "Mei", "Juni", "Juli", "Agustus",
    "September", "Oktober", "Nopember", "Desember"
]

def clean_pesawat(df, tahun):
    # Mengambil nama bulan dari baris ketiga
    bulan = df.iloc[2].astype(str).str.strip()

    # Mengambil kolom Januari sampai Desember
    posisi_bulan = [
        i for i, nama in enumerate(bulan)
        if nama in daftar_bulan
    ]

    assert len(posisi_bulan) == 12, f"Data bulan {tahun} tidak lengkap"

    # Membuat dataframe baru
    df_clean = pd.DataFrame({
        "bulan": bulan.iloc[posisi_bulan].values,
        "penumpang": df.iloc[3, posisi_bulan].values
    })

    # Menambahkan tahun dan mengubah tipe data
    df_clean["tahun"] = tahun
    df_clean["penumpang"] = pd.to_numeric(df_clean["penumpang"])

    return df_clean.reset_index(drop=True)


# Membersihkan data 2020 dan 2021
pesawat_2020 = clean_pesawat(df_pesawat_2020, 2020)
pesawat_2021 = clean_pesawat(df_pesawat_2021, 2021)

# Menggabungkan kedua dataframe
pesawat_clean = pd.concat(
    [pesawat_2020, pesawat_2021],
    ignore_index=True
)

display(pesawat_clean)

,bulan,penumpang,tahun
0,Januari,174381,2020
1,Februari,204459,2020
2,Maret,139860,2020
3,April,30620,2020
4,Mei,1657,2020
5,Juni,16147,2020
6,Juli,38496,2020
7,Agustus,43168,2020
8,September,48207,2020
9,Oktober,59601,2020


In [93]:
# ======================
# Validasi Data Penerbangan
# ======================

# Memeriksa jumlah baris dan missing values
print("Jumlah baris:", len(pesawat_clean))
print("\nMissing values:")
print(pesawat_clean.isna().sum())

# Memeriksa kelengkapan bulan
print("\nJumlah bulan per tahun:")
print(pesawat_clean.groupby("tahun")["bulan"].nunique())

# Memeriksa total penumpang
print("\nTotal penumpang per tahun:")
print(pesawat_clean.groupby("tahun")["penumpang"].sum())

# Validasi jumlah baris, bulan, dan duplikasi
assert len(pesawat_clean) == 24
assert pesawat_clean.isna().sum().sum() == 0
assert pesawat_clean[["tahun", "bulan"]].duplicated().sum() == 0

for tahun, df in pesawat_clean.groupby("tahun"):
    assert set(df["bulan"]) == set(daftar_bulan)

# Membandingkan hasil cleaning dengan total tahunan di Excel
for tahun, df_asli, df_clean in [
    (2020, df_pesawat_2020, pesawat_2020),
    (2021, df_pesawat_2021, pesawat_2021)
]:
    header = df_asli.iloc[2].astype(str).str.strip()
    posisi_total = [
        i for i, nama in enumerate(header)
        if nama == "Tahunan"
    ]

    assert len(posisi_total) == 1, f"Kolom Tahunan {tahun} tidak ditemukan"

    total_asli = pd.to_numeric(df_asli.iloc[3, posisi_total[0]])
    total_clean = df_clean["penumpang"].sum()

    assert total_asli == total_clean, f"Total penumpang {tahun} tidak sesuai"

    print(f"Total {tahun} sesuai dengan data asli: {total_clean:,.0f}")

print("\nValidasi berhasil!")

Jumlah baris: 24

Missing values:
bulan        0
penumpang    0
tahun        0
dtype: int64

Jumlah bulan per tahun:
tahun
2020    12
2021    12
Name: bulan, dtype: int64

Total penumpang per tahun:
tahun
2020    905343
2021    644786
Name: penumpang, dtype: int64
Total 2020 sesuai dengan data asli: 905,343
Total 2021 sesuai dengan data asli: 644,786

Validasi berhasil!


In [94]:
import requests

url = "https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/3300/var/1793/th/120/key/af4ce62a5db0a90cd42ef5200efea09d"

r = requests.get(url, timeout=30)
r.raise_for_status()

data2020 = r.json()
data2020

{'status': 'OK',
 'data-availability': 'available',
 'last_update': '2026-11-02 16:15:05',
 'subject': [{'val': 560, 'label': 'Transportasi'}],
 'var': [{'val': 1793,
   'label': 'Jumlah Debarkasi Penumpang Angkutan Laut di Jawa Tengah',
   'unit': 'Orang',
   'subj': 'Transportasi',
   'def': '',
   'decimal': 0,
   'note': ''}],
 'turvar': [{'val': '0', 'label': 'Tidak ada'}],
 'labelvervar': 'Jumlah Debarkasi Penumpang',
 'vervar': [{'val': 1, 'label': 'Jumlah Penumpang'}],
 'tahun': [{'val': 120, 'label': '2020'}],
 'turtahun': [{'val': 90, 'label': 'Januari'},
  {'val': 91, 'label': 'Februari'},
  {'val': 92, 'label': 'Maret'},
  {'val': 93, 'label': 'April'},
  {'val': 94, 'label': 'Mei'},
  {'val': 95, 'label': 'Juni'},
  {'val': 96, 'label': 'Juli'},
  {'val': 97, 'label': 'Agustus'},
  {'val': 98, 'label': 'September'},
  {'val': 99, 'label': 'Oktober'},
  {'val': 100, 'label': 'Nopember'},
  {'val': 101, 'label': 'Desember'},
  {'val': 102, 'label': 'Tahunan'}],
 'datacontent

In [95]:
url2 = "https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/3300/var/1793/th/121/key/af4ce62a5db0a90cd42ef5200efea09d"

r2 = requests.get(url2, timeout=30)
r2.raise_for_status()

data2021 = r2.json()
data2021

{'status': 'OK',
 'data-availability': 'available',
 'last_update': '2026-11-02 16:15:05',
 'subject': [{'val': 560, 'label': 'Transportasi'}],
 'var': [{'val': 1793,
   'label': 'Jumlah Debarkasi Penumpang Angkutan Laut di Jawa Tengah',
   'unit': 'Orang',
   'subj': 'Transportasi',
   'def': '',
   'decimal': 0,
   'note': ''}],
 'turvar': [{'val': '0', 'label': 'Tidak ada'}],
 'labelvervar': 'Jumlah Debarkasi Penumpang',
 'vervar': [{'val': 1, 'label': 'Jumlah Penumpang'}],
 'tahun': [{'val': 121, 'label': '2021'}],
 'turtahun': [{'val': 90, 'label': 'Januari'},
  {'val': 91, 'label': 'Februari'},
  {'val': 92, 'label': 'Maret'},
  {'val': 93, 'label': 'April'},
  {'val': 94, 'label': 'Mei'},
  {'val': 95, 'label': 'Juni'},
  {'val': 96, 'label': 'Juli'},
  {'val': 97, 'label': 'Agustus'},
  {'val': 98, 'label': 'September'},
  {'val': 99, 'label': 'Oktober'},
  {'val': 100, 'label': 'Nopember'},
  {'val': 101, 'label': 'Desember'},
  {'val': 102, 'label': 'Tahunan'}],
 'datacontent

In [96]:
# ======================
# Cleaning Data Angkutan Laut
# ======================

def extract_df_fixed(data_json):
    # Mengambil tahun
    tahun = int(data_json["tahun"][0]["label"])

    # Membentuk prefix kode BPS
    prefix = (
        str(data_json["vervar"][0]["val"])
        + str(data_json["var"][0]["val"])
        + str(data_json["turvar"][0]["val"])
        + str(data_json["tahun"][0]["val"])
    )

    records = []

    for item in data_json["turtahun"]:
        bulan = item["label"]
        kode = item["val"]

        # Melewati data total tahunan
        if bulan == "Tahunan":
            continue

        key = prefix + str(kode)

        records.append({
            "tahun": tahun,
            "bulan": bulan,
            "penumpang": data_json["datacontent"][key]
        })

    return pd.DataFrame(records)


# Cleaning data 2020 dan 2021
laut_2020 = extract_df_fixed(data2020)
laut_2021 = extract_df_fixed(data2021)

# Menggabungkan data
debarkasi_angkutan_laut = pd.concat(
    [laut_2020, laut_2021],
    ignore_index=True
)

display(debarkasi_angkutan_laut)

,tahun,bulan,penumpang
0,2020,Januari,18821
1,2020,Februari,14667
2,2020,Maret,17103
3,2020,April,5024
4,2020,Mei,2267
5,2020,Juni,6527
6,2020,Juli,7977
7,2020,Agustus,9174
8,2020,September,9272
9,2020,Oktober,13432


In [97]:
# ======================
# Validasi Data Angkutan Laut
# ======================

# Memeriksa jumlah baris dan missing values
print("Jumlah baris:", len(debarkasi_angkutan_laut))

print("\nMissing values:")
print(debarkasi_angkutan_laut.isna().sum())

print("\nJumlah bulan per tahun:")
print(debarkasi_angkutan_laut.groupby("tahun")["bulan"].nunique())

print("\nTotal penumpang per tahun:")
print(debarkasi_angkutan_laut.groupby("tahun")["penumpang"].sum())

# Validasi data
assert len(debarkasi_angkutan_laut) == 24
assert debarkasi_angkutan_laut.isna().sum().sum() == 0
assert debarkasi_angkutan_laut[["tahun", "bulan"]].duplicated().sum() == 0

for tahun, df in debarkasi_angkutan_laut.groupby("tahun"):
    assert set(df["bulan"]) == set(daftar_bulan)

# Membandingkan total bulanan dengan total tahunan BPS
for data_json, df in [
    (data2020, laut_2020),
    (data2021, laut_2021)
]:
    tahun = int(data_json["tahun"][0]["label"])

    prefix = (
        str(data_json["vervar"][0]["val"])
        + str(data_json["var"][0]["val"])
        + str(data_json["turvar"][0]["val"])
        + str(data_json["tahun"][0]["val"])
    )

    total_tahunan = data_json["datacontent"][prefix + "102"]

    assert df["penumpang"].sum() == total_tahunan, (
        f"Total penumpang {tahun} tidak sesuai"
    )

    print(f"Total {tahun} sesuai dengan BPS: {total_tahunan:,}")

print("\nValidasi berhasil!")

Jumlah baris: 24

Missing values:
tahun        0
bulan        0
penumpang    0
dtype: int64

Jumlah bulan per tahun:
tahun
2020    12
2021    12
Name: bulan, dtype: int64

Total penumpang per tahun:
tahun
2020    133604
2021    152016
Name: penumpang, dtype: int64
Total 2020 sesuai dengan BPS: 133,604
Total 2021 sesuai dengan BPS: 152,016

Validasi berhasil!


In [98]:
# ======================
# Mengambil Data Kecelakaan Lalu Lintas
# ======================

file_laka = folder_data / "banyaknya-kecelakaan-lalu-lintas.xls"

# Memeriksa sheet yang tersedia
excel_laka = pd.ExcelFile(file_laka)

print("Sheet tersedia:", excel_laka.sheet_names)

# Membaca data asli tanpa mengubah header
df_raw = pd.read_excel(
    file_laka,
    sheet_name=0,
    header=None
)

print("Ukuran data:", df_raw.shape)

display(df_raw.head(12))

Sheet tersedia: ['Tabel 4.10.15']
Ukuran data: (43, 23)


,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,"Banyaknya Kecelakaan Lalu Lintas, Korban dan N...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Number of Traffic Accidents, Victims and Loss ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Wilayah,NaN,NaN,Jumlah Kecelakaan,NaN,NaN,NaN,Korban,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Kerugian Material (Ribu Rupiah),NaN,NaN,NaN
3,NaN,NaN,NaN,Number of Accident,NaN,NaN,NaN,Victim,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Region,NaN,NaN,NaN,NaN,NaN,NaN,Meninggal,NaN,NaN,...,NaN,NaN,Luka Ringan,NaN,NaN,NaN,Loss (Thousand Rupiah),NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Death,NaN,NaN,...,NaN,NaN,Easy Wound,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,2018,2019.0,2020.0,2021.0,2018,2019.0,2020.0,...,2020.0,2021.0,2018,2019.0,2020.0,2021.0,2018,2019.0,2020.0,2.021000e+03
7,01.,Res. Cilacap,NaN,753,898.0,851.0,867.0,208,189.0,172.0,...,1.0,2.0,1147,1263.0,1035.0,1057.0,709300000,874150000.0,620700000.0,7.146000e+08
8,02.,Res. Banyumas,NaN,940,1050.0,1574.0,1595.0,220,224.0,217.0,...,1.0,13.0,1030,1149.0,1765.0,1816.0,609790000,685580000.0,952820000.0,1.121220e+09
9,03.,Res. Purbalingga,NaN,498,579.0,439.0,443.0,104,77.0,75.0,...,0.0,0.0,633,715.0,497.0,508.0,280700000,344600000.0,239100000.0,1.875000e+08


In [99]:
# ======================
# Menyeleksi Data Kecelakaan
# ======================

# df_raw merupakan hasil pd.read_excel(..., header=None)

# Ambil data setelah baris header
df_data = df_raw.iloc[7:].copy()

# Ambil hanya baris yang memiliki kode wilayah
mask_wilayah = (
    df_data.iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.match(r"^\d{1,2}\.$")
)

df_data = df_data.loc[mask_wilayah].reset_index(drop=True)

# Ambil kode wilayah sebagai penanda kabupaten/kota
df_data["kode_wilayah"] = (
    df_data.iloc[:, 0]
    .astype(str)
    .str.replace(".", "", regex=False)
    .astype(int)
)

# Memastikan kode wilayah lengkap
assert sorted(df_data["kode_wilayah"].tolist()) == list(range(1, 36))

print("Jumlah wilayah:", len(df_data))
display(df_data[[0, 1, "kode_wilayah"]])

Jumlah wilayah: 35


,0,1,kode_wilayah
0,01.,Res. Cilacap,1
1,02.,Res. Banyumas,2
2,03.,Res. Purbalingga,3
3,04.,Res. Banjarnegara,4
4,05.,Res. Kebumen,5
5,06.,Res. Purworejo,6
6,07.,Res. Wonosobo,7
7,08.,Res. Magelang,8
8,09.,Res. Boyolali,9
9,10.,Res. Klaten,10


In [100]:
# ======================
# Membuat DataFrame Kecelakaan
# ======================

df_clean = pd.DataFrame({
    "kabupaten_kota": df_data.iloc[:, 1],

    "jumlah_kecelakaan_2020": df_data.iloc[:, 5],
    "jumlah_kecelakaan_2021": df_data.iloc[:, 6],

    "meninggal_2020": df_data.iloc[:, 9],
    "meninggal_2021": df_data.iloc[:, 10],

    "luka_berat_2020": df_data.iloc[:, 13],
    "luka_berat_2021": df_data.iloc[:, 14],

    "luka_ringan_2020": df_data.iloc[:, 17],
    "luka_ringan_2021": df_data.iloc[:, 18],

    "kerugian_2020": df_data.iloc[:, 21],
    "kerugian_2021": df_data.iloc[:, 22]
})

# Menyimpan kode wilayah sementara untuk cleaning
df_clean["kode_wilayah"] = df_data["kode_wilayah"]

display(df_clean.head())

,kabupaten_kota,jumlah_kecelakaan_2020,jumlah_kecelakaan_2021,meninggal_2020,meninggal_2021,luka_berat_2020,luka_berat_2021,luka_ringan_2020,luka_ringan_2021,kerugian_2020,kerugian_2021,kode_wilayah
0,Res. Cilacap,851.0,867.0,172.0,183.0,1.0,2.0,1035.0,1057.0,620700000.0,7.146000e+08,1
1,Res. Banyumas,1574.0,1595.0,217.0,216.0,1.0,13.0,1765.0,1816.0,952820000.0,1.121220e+09,2
2,Res. Purbalingga,439.0,443.0,75.0,56.0,0.0,0.0,497.0,508.0,239100000.0,1.875000e+08,3
3,Res. Banjarnegara,451.0,482.0,75.0,60.0,4.0,5.0,528.0,598.0,518650000.0,3.885000e+08,4
4,Res. Kebumen,618.0,649.0,121.0,129.0,0.0,0.0,732.0,760.0,891200000.0,1.576850e+08,5


In [101]:
# ======================
# Cleaning Data Kecelakaan
# ======================

df_clean2 = df_clean.copy()

# Membersihkan awalan nama wilayah
df_clean2["kabupaten_kota"] = (
    df_clean2["kabupaten_kota"]
    .astype(str)
    .str.replace(
        r"^(ResTabes|ResTa|Res)\.?\s*",
        "",
        regex=True
    )
    .str.strip()
)

# Membedakan kabupaten dan kota berdasarkan kode wilayah
# Kode 01-29: Kabupaten
# Kode 30-35: Kota
mask_kota = df_clean2["kode_wilayah"].between(30, 35)

df_clean2.loc[mask_kota, "kabupaten_kota"] = (
    "Kota " + df_clean2.loc[mask_kota, "kabupaten_kota"]
)

# Mengubah kolom indikator menjadi numerik
kolom_nilai = [
    col for col in df_clean2.columns
    if col not in ["kabupaten_kota", "kode_wilayah"]
]

for col in kolom_nilai:
    df_clean2[col] = pd.to_numeric(
        df_clean2[col],
        errors="raise"
    )

# Konversi kerugian dari ribu rupiah ke juta rupiah
for col in ["kerugian_2020", "kerugian_2021"]:
    df_clean2[col] = df_clean2[col] / 1_000_000
# Hapus kode wilayah sementara
df_clean2 = df_clean2.drop(columns=["kode_wilayah"])

# Data akhir
laka_lantas = df_clean2.reset_index(drop=True)

display(laka_lantas)

,kabupaten_kota,jumlah_kecelakaan_2020,jumlah_kecelakaan_2021,meninggal_2020,meninggal_2021,luka_berat_2020,luka_berat_2021,luka_ringan_2020,luka_ringan_2021,kerugian_2020,kerugian_2021
0,Cilacap,851.0,867.0,172.0,183.0,1.0,2.0,1035.0,1057.0,620.700,714.600
1,Banyumas,1574.0,1595.0,217.0,216.0,1.0,13.0,1765.0,1816.0,952.820,1121.220
2,Purbalingga,439.0,443.0,75.0,56.0,0.0,0.0,497.0,508.0,239.100,187.500
3,Banjarnegara,451.0,482.0,75.0,60.0,4.0,5.0,528.0,598.0,518.650,388.500
4,Kebumen,618.0,649.0,121.0,129.0,0.0,0.0,732.0,760.0,891.200,157.685
5,Purworejo,505.0,482.0,87.0,98.0,0.0,0.0,561.0,540.0,373.200,357.300
6,Wonosobo,276.0,298.0,53.0,62.0,2.0,2.0,319.0,316.0,186.765,187.500
7,Magelang,782.0,827.0,135.0,164.0,2.0,2.0,941.0,912.0,254.050,319.920
8,Boyolali,761.0,838.0,91.0,109.0,1.0,1.0,1087.0,1335.0,278.400,250.770
9,Klaten,1003.0,1260.0,142.0,148.0,7.0,5.0,1712.0,2029.0,1153.540,1314.070


In [102]:
# ======================
# Validasi Data Kecelakaan
# ======================

print("Jumlah baris:", len(laka_lantas))
print("Jumlah wilayah:", laka_lantas["kabupaten_kota"].nunique())

print("\nMissing values:")
print(laka_lantas.isna().sum())

print("\nDaftar kota:")
print(laka_lantas.loc[
    laka_lantas["kabupaten_kota"].str.startswith("Kota "),
    "kabupaten_kota"
].tolist())

# Validasi jumlah wilayah dan kolom
assert laka_lantas.shape == (35, 11)

# Validasi nama wilayah
assert laka_lantas["kabupaten_kota"].is_unique
assert laka_lantas["kabupaten_kota"].notna().all()

# Validasi jumlah kabupaten dan kota
assert (
    laka_lantas["kabupaten_kota"].str.startswith("Kota ").sum()
    == 6
)

# Memastikan empat pasangan wilayah tetap berbeda
for nama in ["Magelang", "Semarang", "Pekalongan", "Tegal"]:
    assert nama in laka_lantas["kabupaten_kota"].values
    assert f"Kota {nama}" in laka_lantas["kabupaten_kota"].values

# Validasi indikator
assert laka_lantas[kolom_nilai].notna().all().all()
assert (laka_lantas[kolom_nilai] >= 0).all().all()

print("\nValidasi berhasil!")

Jumlah baris: 35
Jumlah wilayah: 35

Missing values:
kabupaten_kota            0
jumlah_kecelakaan_2020    0
jumlah_kecelakaan_2021    0
meninggal_2020            0
meninggal_2021            0
luka_berat_2020           0
luka_berat_2021           0
luka_ringan_2020          0
luka_ringan_2021          0
kerugian_2020             0
kerugian_2021             0
dtype: int64

Daftar kota:
['Kota Magelang', 'Kota Surakarta', 'Kota Salatiga', 'Kota Semarang', 'Kota Pekalongan', 'Kota Tegal']

Validasi berhasil!


In [103]:
# ======================
# Mengambil Data Panjang Jalan
# ======================

file_jalan_2020 = folder_data / "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2020.csv"

file_jalan_2021 = folder_data / "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2021.csv"

# Membaca data asli
jalan_2020 = pd.read_csv(
    file_jalan_2020,
    sep=None,
    engine="python",
    header=None
)

jalan_2021 = pd.read_csv(
    file_jalan_2021,
    sep=None,
    engine="python",
    header=None
)

print("Ukuran data 2020:", jalan_2020.shape)
print("Ukuran data 2021:", jalan_2021.shape)

display(jalan_2020.head(12))
display(jalan_2021.head(12))

Ukuran data 2020: (37, 5)
Ukuran data 2021: (37, 5)


,0,1,2,3,4
0,﻿Kabupaten/Kota,Panjang Jalan Negara (km) (Km),Panjang Jalan Provinsi (km) (Km),Panjang Jalan Kabupaten/Kota (km) (Km),Jumlah Panjang Jalan(km) (Km)
1,Cilacap,174,83,1269,1526
2,Banyumas,171,40,1508,1719
3,Purbalingga,0,34,888,922
4,Banjarnegara,59,88,939,1086
5,Kebumen,56,30,960,1046
6,Purworejo,37,103,769,909
7,Wonosobo,30,111,999,1141
8,Magelang,38,119,1001,1157
9,Boyolali,33,52,678,763


,0,1,2,3,4
0,﻿Kabupaten/Kota,Panjang Jalan Negara (km) (Km),Panjang Jalan Provinsi (km) (Km),Panjang Jalan Kabupaten/Kota (km) (Km),Jumlah Panjang Jalan(km) (Km)
1,Cilacap,174,77,1269,1519
2,Banyumas,171,40,1508,1719
3,Purbalingga,NaN,34,888,922
4,Banjarnegara,59,88,939,1086
5,Kebumen,56,30,960,1046
6,Purworejo,37,103,769,909
7,Wonosobo,30,111,999,1141
8,Magelang,38,119,1001,1157
9,Boyolali,33,49,678,760


In [104]:
# ======================
# Cleaning Data Panjang Jalan
# ======================

def clean_jalan(df, tahun):
    # Mengambil 35 baris kabupaten/kota setelah header
    df_clean = df.iloc[1:36, :5].copy()

    # Mengubah nama kolom
    df_clean.columns = [
        "kabupaten_kota",
        "jalan_negara",
        "jalan_provinsi",
        "jalan_kabupaten_kota",
        "total_panjang_jalan"
    ]

    # Membersihkan nama wilayah
    df_clean["kabupaten_kota"] = (
        df_clean["kabupaten_kota"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Mengubah kolom panjang jalan menjadi numerik
    kolom_nilai = [
        "jalan_negara",
        "jalan_provinsi",
        "jalan_kabupaten_kota",
        "total_panjang_jalan"
    ]

    for col in kolom_nilai:
        df_clean[col] = pd.to_numeric(
            df_clean[col],
            errors="coerce"
        )

    # Menambahkan tahun
    df_clean["tahun"] = tahun

    return df_clean.reset_index(drop=True)


# Cleaning data 2020 dan 2021
jalan_clean_2020 = clean_jalan(jalan_2020, 2020)
jalan_clean_2021 = clean_jalan(jalan_2021, 2021)

# Menggabungkan data
jalan_clean = pd.concat(
    [jalan_clean_2020, jalan_clean_2021],
    ignore_index=True
)

display(jalan_clean)

,kabupaten_kota,jalan_negara,jalan_provinsi,jalan_kabupaten_kota,total_panjang_jalan,tahun
0,Cilacap,174.0,83.0,1269,1526,2020
1,Banyumas,171.0,40.0,1508,1719,2020
2,Purbalingga,0.0,34.0,888,922,2020
3,Banjarnegara,59.0,88.0,939,1086,2020
4,Kebumen,56.0,30.0,960,1046,2020
...,...,...,...,...,...,...
65,Kota Surakarta,14.0,9.0,214,238,2021
66,Kota Salatiga,12.0,NaN,337,350,2021
67,Kota Semarang,68.0,40.0,840,948,2021
68,Kota Pekalongan,NaN,NaN,153,153,2021


In [105]:
# ======================
# Validasi Data Panjang Jalan
# ======================

print("Jumlah baris:", len(jalan_clean))

print("\nJumlah data per tahun:")
print(jalan_clean.groupby("tahun").size())

print("\nJumlah wilayah unik:")
print(jalan_clean.groupby("tahun")["kabupaten_kota"].nunique())

print("\nMissing values:")
print(jalan_clean.isna().sum())

# Memeriksa enam kota di bagian akhir
print("\nWilayah bagian akhir:")
display(jalan_clean.groupby("tahun").tail(6))

# Memeriksa kecocokan total panjang jalan
jalan_clean["total_hitung"] = jalan_clean[
    ["jalan_negara", "jalan_provinsi", "jalan_kabupaten_kota"]
].sum(axis=1, min_count=1)

jalan_clean["selisih"] = (
    jalan_clean["total_panjang_jalan"]
    - jalan_clean["total_hitung"]
)

print("\nData dengan selisih lebih dari 1 km:")
display(jalan_clean.loc[jalan_clean["selisih"].abs() > 1])

# Validasi jumlah wilayah
assert len(jalan_clean) == 70
assert (jalan_clean.groupby("tahun").size() == 35).all()
assert (jalan_clean.groupby("tahun")["kabupaten_kota"].nunique() == 35).all()

# Validasi data
assert jalan_clean["kabupaten_kota"].notna().all()
assert jalan_clean["total_panjang_jalan"].notna().all()
assert (jalan_clean["total_panjang_jalan"] > 0).all()

assert not jalan_clean.duplicated(
    subset=["kabupaten_kota", "tahun"]
).any()

print("\nValidasi berhasil!")

Jumlah baris: 70

Jumlah data per tahun:
tahun
2020    35
2021    35
dtype: int64

Jumlah wilayah unik:
tahun
2020    35
2021    35
Name: kabupaten_kota, dtype: int64

Missing values:
kabupaten_kota          0
jalan_negara            3
jalan_provinsi          4
jalan_kabupaten_kota    0
total_panjang_jalan     0
tahun                   0
dtype: int64

Wilayah bagian akhir:


,kabupaten_kota,jalan_negara,jalan_provinsi,jalan_kabupaten_kota,total_panjang_jalan,tahun
29,Kota Magelang,8.0,0.0,119,127,2020
30,Kota Surakarta,14.0,9.0,214,238,2020
31,Kota Salatiga,12.0,0.0,337,350,2020
32,Kota Semarang,68.0,27.0,840,935,2020
33,Kota Pekalongan,11.0,4.0,153,153,2020
34,Kota Tegal,13.0,0.0,231,244,2020
64,Kota Magelang,8.0,NaN,119,127,2021
65,Kota Surakarta,14.0,9.0,214,238,2021
66,Kota Salatiga,12.0,NaN,337,350,2021
67,Kota Semarang,68.0,40.0,840,948,2021



Data dengan selisih lebih dari 1 km:


,kabupaten_kota,jalan_negara,jalan_provinsi,jalan_kabupaten_kota,total_panjang_jalan,tahun,total_hitung,selisih
33,Kota Pekalongan,11.0,4.0,153,153,2020,168.0,-15.0



Validasi berhasil!


In [106]:
# ======================
# Memeriksa Ketidaksesuaian Data Panjang Jalan
# ======================

# Memeriksa data Kota Pekalongan dari CSV asli
print("Data asli 2020:")
display(
    jalan_2020.loc[
        jalan_2020[0].astype(str).str.contains(
            "Pekalongan", case=False, na=False
        )
    ]
)

# Membandingkan data hasil cleaning
print("\nData hasil cleaning:")
display(
    jalan_clean.loc[
        jalan_clean["kabupaten_kota"].str.contains(
            "Pekalongan", case=False, na=False
        )
    ]
)

# Memeriksa baris dengan komponen jalan yang kosong
print("\nData dengan missing values:")
display(
    jalan_clean.loc[
        jalan_clean[["jalan_negara", "jalan_provinsi"]]
        .isna().any(axis=1)
    ]
)

Data asli 2020:


,0,1,2,3,4
26,Pekalongan,9,86,670,765
34,Kota Pekalongan,11,4,153,153



Data hasil cleaning:


,kabupaten_kota,jalan_negara,jalan_provinsi,jalan_kabupaten_kota,total_panjang_jalan,tahun,total_hitung,selisih
25,Pekalongan,9.0,86.0,670,765,2020,765.0,0.0
33,Kota Pekalongan,11.0,4.0,153,153,2020,168.0,-15.0
60,Pekalongan,9.0,85.0,670,764,2021,764.0,0.0
68,Kota Pekalongan,NaN,NaN,153,153,2021,153.0,0.0



Data dengan missing values:


,kabupaten_kota,jalan_negara,jalan_provinsi,jalan_kabupaten_kota,total_panjang_jalan,tahun,total_hitung,selisih
37,Purbalingga,NaN,34.0,888,922,2021,922.0,0.0
49,Grobogan,NaN,189.0,918,1107,2021,1107.0,0.0
64,Kota Magelang,8.0,NaN,119,127,2021,127.0,0.0
66,Kota Salatiga,12.0,NaN,337,350,2021,349.0,1.0
68,Kota Pekalongan,NaN,NaN,153,153,2021,153.0,0.0
69,Kota Tegal,13.0,NaN,231,244,2021,244.0,0.0


In [107]:
# ======================
# Mengambil Data Kendaraan Bermotor
# ======================

file_kendaraan_2020 = folder_data / "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2020.csv"
file_kendaraan_2021 = folder_data / "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2021.csv"

# Membaca data asli
kendaraan_2020 = pd.read_csv(file_kendaraan_2020)
kendaraan_2021 = pd.read_csv(file_kendaraan_2021)

print("Ukuran data 2020:", kendaraan_2020.shape)
print("Ukuran data 2021:", kendaraan_2021.shape)

display(kendaraan_2020.head(10))
display(kendaraan_2020.tail(10))

Ukuran data 2020: (39, 6)
Ukuran data 2021: (39, 6)


,Kabupaten/Kota,Jumlah Kendaraan Bermotor - Mobil Penumpang,Jumlah Kendaraan Bermotor - Bus,Jumlah Kendaraan Bermotor - Truk,Jumlah Kendaraan Bermotor - Sepeda Motor,Jumlah Kendaraan Bermotor - Jumlah
0,Cilacap,44767.0,793.0,20762.0,752008.0,818330.0
1,Banyumas,60915.0,1721.0,25160.0,693638.0,781434.0
2,Purbalingga,23347.0,397.0,12106.0,361588.0,397438.0
3,Banjarnegara,20088.0,648.0,13587.0,320372.0,354695.0
4,Kebumen,25038.0,1224.0,14191.0,494111.0,534564.0
5,Purworejo,21659.0,776.0,8573.0,297285.0,328293.0
6,Wonosobo,20135.0,983.0,14224.0,265486.0,300828.0
7,Magelang,43327.0,1010.0,17643.0,454980.0,516960.0
8,Boyolali,38230.0,1719.0,17217.0,469412.0,526578.0
9,Klaten,55235.0,1278.0,22415.0,710411.0,789339.0


,Kabupaten/Kota,Jumlah Kendaraan Bermotor - Mobil Penumpang,Jumlah Kendaraan Bermotor - Bus,Jumlah Kendaraan Bermotor - Truk,Jumlah Kendaraan Bermotor - Sepeda Motor,Jumlah Kendaraan Bermotor - Jumlah
29,Kota Magelang,19151.0,676.0,10307.0,369753.0,399887.0
30,Kota Surakarta,71120.0,1802.0,23457.0,465626.0,562005.0
31,Kota Salatiga,19701.0,471.0,5733.0,120507.0,146412.0
32,Kota Semarang,231164.0,3059.0,76570.0,1382434.0,1693227.0
33,Kota Pekalongan,15436.0,614.0,6534.0,175182.0,197766.0
34,Kota Tegal,12832.0,846.0,5985.0,162965.0,182612.0
35,Jawa Tengah,1309343.0,37785.0,580411.0,16214173.0,18141696.0
36,NaN,NaN,NaN,NaN,NaN,NaN
37,Catatan,NaN,NaN,NaN,NaN,NaN
38,Data yang tersedia dari DITLANTAS POLDA SEMARA...,NaN,NaN,NaN,NaN,NaN


In [108]:
# ======================
# Cleaning Data Kendaraan Bermotor
# ======================

# Mengubah nama kolom
rename_cols = {
    "Kabupaten/Kota": "kabupaten_kota",
    "Jumlah Kendaraan Bermotor - Mobil Penumpang": "mobil_penumpang",
    "Jumlah Kendaraan Bermotor - Bus": "bus",
    "Jumlah Kendaraan Bermotor - Truk": "truk",
    "Jumlah Kendaraan Bermotor - Sepeda Motor": "sepeda_motor",
    "Jumlah Kendaraan Bermotor - Jumlah": "total"
}

# Kolom yang berisi jumlah kendaraan
kolom_nilai = [
    "mobil_penumpang",
    "bus",
    "truk",
    "sepeda_motor",
    "total"
]

# Mengambil daftar wilayah dari jalan_clean
daftar_wilayah = set(
    jalan_clean["kabupaten_kota"].astype(str).str.strip()
)

def clean_kendaraan(df, tahun):
    # Membuat salinan data
    df_clean = df.copy()

    # Mengubah nama kolom
    df_clean = df_clean.rename(columns=rename_cols)

    # Membersihkan nama wilayah
    df_clean["kabupaten_kota"] = (
        df_clean["kabupaten_kota"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Mengambil hanya 35 kabupaten/kota
    df_clean = df_clean[
        df_clean["kabupaten_kota"].isin(daftar_wilayah)
    ].copy()

    # Mengubah kolom jumlah kendaraan menjadi numerik
    for col in kolom_nilai:
        df_clean[col] = pd.to_numeric(
            df_clean[col],
            errors="coerce"
        )

    # Menambahkan tahun
    df_clean["tahun"] = tahun

    # Memilih kolom yang diperlukan
    df_clean = df_clean[
        ["kabupaten_kota"] + kolom_nilai + ["tahun"]
    ]

    return df_clean.reset_index(drop=True)


# Cleaning data 2020 dan 2021
kendaraan_clean_2020 = clean_kendaraan(kendaraan_2020, 2020)
kendaraan_clean_2021 = clean_kendaraan(kendaraan_2021, 2021)

# Menggabungkan data
kendaraan_bermotor = pd.concat(
    [kendaraan_clean_2020, kendaraan_clean_2021],
    ignore_index=True
)

display(kendaraan_bermotor)

,kabupaten_kota,mobil_penumpang,bus,truk,sepeda_motor,total,tahun
0,Cilacap,44767.0,793.0,20762.0,752008.0,818330.0,2020
1,Banyumas,60915.0,1721.0,25160.0,693638.0,781434.0,2020
2,Purbalingga,23347.0,397.0,12106.0,361588.0,397438.0,2020
3,Banjarnegara,20088.0,648.0,13587.0,320372.0,354695.0,2020
4,Kebumen,25038.0,1224.0,14191.0,494111.0,534564.0,2020
...,...,...,...,...,...,...,...
65,Kota Surakarta,287764.0,1300.0,46197.0,509927.0,845188.0,2021
66,Kota Salatiga,23381.0,300.0,5014.0,63276.0,91971.0,2021
67,Kota Semarang,281971.0,3539.0,78037.0,1512234.0,1875781.0,2021
68,Kota Pekalongan,12810.0,407.0,7094.0,201732.0,222043.0,2021


In [109]:
# ======================
# Validasi Data Kendaraan Bermotor
# ======================

# Memeriksa jumlah data
print("Jumlah baris:", len(kendaraan_bermotor))

print("\nJumlah wilayah per tahun:")
print(
    kendaraan_bermotor.groupby("tahun")["kabupaten_kota"].nunique()
)

print("\nMissing values:")
print(kendaraan_bermotor.isna().sum())

# Menghitung jumlah kendaraan berdasarkan jenisnya
kolom_kendaraan = [
    "mobil_penumpang",
    "bus",
    "truk",
    "sepeda_motor"
]

kendaraan_bermotor["total_hitung"] = (
    kendaraan_bermotor[kolom_kendaraan].sum(
        axis=1, min_count=4
    )
)

# Menghitung selisih dengan total dari BPS
kendaraan_bermotor["selisih"] = (
    kendaraan_bermotor["total"]
    - kendaraan_bermotor["total_hitung"]
)

print("\nData dengan total tidak sesuai:")
display(
    kendaraan_bermotor.loc[
        kendaraan_bermotor["selisih"].notna()
        & (kendaraan_bermotor["selisih"].abs() > 0.01)
    ]
)

# Validasi jumlah data
assert len(kendaraan_bermotor) == 70
assert (kendaraan_bermotor.groupby("tahun").size() == 35).all()

# Validasi kesesuaian wilayah
for tahun, df in kendaraan_bermotor.groupby("tahun"):
    assert set(df["kabupaten_kota"]) == daftar_wilayah, (
        f"Wilayah tahun {tahun} tidak sesuai"
    )

# Validasi missing values
kolom_utama = ["kabupaten_kota"] + kolom_nilai + ["tahun"]

assert kendaraan_bermotor[kolom_utama].notna().all().all()

# Validasi duplikasi
assert not kendaraan_bermotor.duplicated(
    subset=["kabupaten_kota", "tahun"]
).any()

# Validasi nilai negatif
assert (kendaraan_bermotor[kolom_nilai] >= 0).all().all()

print("\nValidasi struktur berhasil!")

Jumlah baris: 70

Jumlah wilayah per tahun:
tahun
2020    35
2021    35
Name: kabupaten_kota, dtype: int64

Missing values:
kabupaten_kota     0
mobil_penumpang    0
bus                0
truk               0
sepeda_motor       0
total              0
tahun              0
dtype: int64

Data dengan total tidak sesuai:


,kabupaten_kota,mobil_penumpang,bus,truk,sepeda_motor,total,tahun,total_hitung,selisih
34,Kota Tegal,12832.0,846.0,5985.0,162965.0,182612.0,2020,182628.0,-16.0



Validasi struktur berhasil!


In [110]:
# ======================
# Memeriksa Anomali Data Kendaraan
# ======================

# Membaca ulang data asli 2020
kendaraan_2020_asli = pd.read_csv(file_kendaraan_2020)

# Memeriksa data asli Kota Tegal
print("Data asli Kota Tegal 2020:")

display(
    kendaraan_2020_asli.loc[
        kendaraan_2020_asli.iloc[:, 0]
        .astype(str)
        .str.strip()
        .eq("Kota Tegal")
    ]
)

# Mengambil data yang totalnya tidak sesuai
kendaraan_anomali = kendaraan_bermotor.loc[
    kendaraan_bermotor["selisih"].abs() > 0.01
].copy()

print("\nData dengan total tidak sesuai:")
display(kendaraan_anomali)

print("\nJumlah anomali:", len(kendaraan_anomali))

Data asli Kota Tegal 2020:


,Kabupaten/Kota,Jumlah Kendaraan Bermotor - Mobil Penumpang,Jumlah Kendaraan Bermotor - Bus,Jumlah Kendaraan Bermotor - Truk,Jumlah Kendaraan Bermotor - Sepeda Motor,Jumlah Kendaraan Bermotor - Jumlah
34,Kota Tegal,12832.0,846.0,5985.0,162965.0,182612.0



Data dengan total tidak sesuai:


,kabupaten_kota,mobil_penumpang,bus,truk,sepeda_motor,total,tahun,total_hitung,selisih
34,Kota Tegal,12832.0,846.0,5985.0,162965.0,182612.0,2020,182628.0,-16.0



Jumlah anomali: 1


In [111]:
# ======================
# Validasi Wilayah Antar-Dataset
# ======================

# Mengambil nama wilayah dari setiap dataset
wilayah_laka = set(
    laka_lantas["kabupaten_kota"].astype(str).str.strip()
)

wilayah_jalan = set(
    jalan_clean["kabupaten_kota"].astype(str).str.strip()
)

wilayah_kendaraan = set(
    kendaraan_bermotor["kabupaten_kota"].astype(str).str.strip()
)

# Memeriksa jumlah wilayah
print("Jumlah wilayah kecelakaan:", len(wilayah_laka))
print("Jumlah wilayah panjang jalan:", len(wilayah_jalan))
print("Jumlah wilayah kendaraan:", len(wilayah_kendaraan))

# Memeriksa wilayah yang tidak cocok
print("\nWilayah kecelakaan yang tidak ada di panjang jalan:")
print(sorted(wilayah_laka - wilayah_jalan))

print("\nWilayah panjang jalan yang tidak ada di kecelakaan:")
print(sorted(wilayah_jalan - wilayah_laka))

print("\nWilayah kendaraan yang tidak ada di panjang jalan:")
print(sorted(wilayah_kendaraan - wilayah_jalan))

print("\nWilayah panjang jalan yang tidak ada di kendaraan:")
print(sorted(wilayah_jalan - wilayah_kendaraan))

# Validasi kesesuaian wilayah
assert len(wilayah_laka) == 35
assert len(wilayah_jalan) == 35
assert len(wilayah_kendaraan) == 35

assert wilayah_laka == wilayah_jalan
assert wilayah_jalan == wilayah_kendaraan

print("\nValidasi wilayah berhasil!")

Jumlah wilayah kecelakaan: 35
Jumlah wilayah panjang jalan: 35
Jumlah wilayah kendaraan: 35

Wilayah kecelakaan yang tidak ada di panjang jalan:
[]

Wilayah panjang jalan yang tidak ada di kecelakaan:
[]

Wilayah kendaraan yang tidak ada di panjang jalan:
[]

Wilayah panjang jalan yang tidak ada di kendaraan:
[]

Validasi wilayah berhasil!


# Membuat Dimensi

In [112]:
# ======================
# Membuat Dimensi Tahun
# ======================

dim_tahun = pd.DataFrame({
    "id_tahun": [1, 2],
    "tahun": [2020, 2021]
})

display(dim_tahun)

,id_tahun,tahun
0,1,2020
1,2,2021


In [113]:
# ======================
# Membuat Dimensi Bulan
# ======================

daftar_bulan = [
    "Januari", "Februari", "Maret", "April",
    "Mei", "Juni", "Juli", "Agustus",
    "September", "Oktober", "Nopember", "Desember"
]

dim_bulan = pd.DataFrame({
    "id_bulan": range(1, 13),
    "nama_bulan": daftar_bulan
})

# Menentukan kuartal
dim_bulan["kuartal"] = (
    (dim_bulan["id_bulan"] - 1) // 3 + 1
)

display(dim_bulan)

,id_bulan,nama_bulan,kuartal
0,1,Januari,1
1,2,Februari,1
2,3,Maret,1
3,4,April,2
4,5,Mei,2
5,6,Juni,2
6,7,Juli,3
7,8,Agustus,3
8,9,September,3
9,10,Oktober,4


In [114]:
# ======================
# Membuat Dimensi Lokasi
# ======================

# Mengambil nama wilayah dari data kecelakaan
dim_lokasi = (
    laka_lantas[["kabupaten_kota"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Membersihkan format nama wilayah
dim_lokasi["kabupaten_kota"] = (
    dim_lokasi["kabupaten_kota"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Membuat ID lokasi berdasarkan urutan data
dim_lokasi.insert(
    0, "id_lokasi", range(1, len(dim_lokasi) + 1)
)

display(dim_lokasi)

,id_lokasi,kabupaten_kota
0,1,cilacap
1,2,banyumas
2,3,purbalingga
3,4,banjarnegara
4,5,kebumen
5,6,purworejo
6,7,wonosobo
7,8,magelang
8,9,boyolali
9,10,klaten


In [115]:
# ======================
# Validasi Dimensi Lokasi
# ======================

print("Jumlah wilayah:", len(dim_lokasi))
print("Jumlah wilayah unik:", dim_lokasi["kabupaten_kota"].nunique())

# Memeriksa duplikasi dan missing values
assert len(dim_lokasi) == 35
assert dim_lokasi["kabupaten_kota"].is_unique
assert dim_lokasi.notna().all().all()

# Memeriksa kesesuaian wilayah antar sumber
def normalisasi_lokasi(df):
    return set(
        df["kabupaten_kota"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )

wilayah_dim = set(dim_lokasi["kabupaten_kota"])
wilayah_jalan = normalisasi_lokasi(jalan_clean)
wilayah_kendaraan = normalisasi_lokasi(kendaraan_bermotor)

assert wilayah_dim == wilayah_jalan
assert wilayah_dim == wilayah_kendaraan

# Memeriksa kabupaten dan kota dengan nama sama
for nama in ["magelang", "semarang", "pekalongan", "tegal"]:
    assert nama in wilayah_dim
    assert f"kota_{nama}" in wilayah_dim

print("\nValidasi dimensi lokasi berhasil!")

Jumlah wilayah: 35
Jumlah wilayah unik: 35

Validasi dimensi lokasi berhasil!


In [116]:
# ======================
# Membuat Dimensi Jenis Kendaraan
# ======================

# Mengambil jenis kendaraan dari data yang sudah dibersihkan
jenis_kendaraan = [
    col for col in kendaraan_bermotor.columns
    if col in ["mobil_penumpang", "bus", "truk", "sepeda_motor"]
]

dim_jenis_kendaraan = pd.DataFrame({
    "nama_jenis_kendaraan": jenis_kendaraan
})

# Membuat ID jenis kendaraan
dim_jenis_kendaraan.insert(
    0, "id_jenis_kendaraan",
    range(1, len(dim_jenis_kendaraan) + 1)
)

display(dim_jenis_kendaraan)

,id_jenis_kendaraan,nama_jenis_kendaraan
0,1,mobil_penumpang
1,2,bus
2,3,truk
3,4,sepeda_motor


In [117]:
# ======================
# Membuat Dimensi Jenis Transportasi
# ======================

# Mengambil jenis transportasi berdasarkan sumber data
jenis_transportasi = [
    "Angkutan Laut",
    "Penerbangan Domestik"
]

dim_jenis_transportasi = pd.DataFrame({
    "nama_jenis_transportasi": jenis_transportasi
})

# Membuat ID jenis transportasi
dim_jenis_transportasi.insert(
    0, "id_jenis_transportasi",
    range(1, len(dim_jenis_transportasi) + 1)
)

display(dim_jenis_transportasi)

,id_jenis_transportasi,nama_jenis_transportasi
0,1,Angkutan Laut
1,2,Penerbangan Domestik


In [118]:
# ======================
# Membuat Dimensi Jenis Jalan
# ======================

# Mengambil jenis jalan dari nama kolom
kolom_jalan = [
    "jalan_negara",
    "jalan_provinsi",
    "jalan_kabupaten_kota"
]

jenis_jalan = [
    col.replace("jalan_", "")
    for col in kolom_jalan
    if col in jalan_clean.columns
]

dim_jenis_jalan = pd.DataFrame({
    "nama_jenis_jalan": jenis_jalan
})

# Membuat ID jenis jalan
dim_jenis_jalan.insert(
    0, "id_jenis_jalan",
    range(1, len(dim_jenis_jalan) + 1)
)

display(dim_jenis_jalan)

,id_jenis_jalan,nama_jenis_jalan
0,1,negara
1,2,provinsi
2,3,kabupaten_kota


In [119]:
# ======================
# Membuat Dimensi Indikator Kecelakaan
# ======================

# Mengambil indikator dari nama kolom data kecelakaan
kolom_indikator = [
    col for col in laka_lantas.columns
    if col.endswith(("_2020", "_2021"))
]

# Menghapus akhiran tahun dan mengambil indikator unik
indikator_kecelakaan = sorted(set(
    col.rsplit("_", 1)[0]
    for col in kolom_indikator
))

dim_indikator_kecelakaan = pd.DataFrame({
    "nama_indikator": indikator_kecelakaan
})

# Membuat ID indikator
dim_indikator_kecelakaan.insert(
    0, "id_indikator",
    range(1, len(dim_indikator_kecelakaan) + 1)
)

display(dim_indikator_kecelakaan)

,id_indikator,nama_indikator
0,1,jumlah_kecelakaan
1,2,kerugian
2,3,luka_berat
3,4,luka_ringan
4,5,meninggal


In [120]:
# ======================
# Validasi Seluruh Dimensi
# ======================

daftar_dimensi = {
    "dim_tahun": (dim_tahun, 2),
    "dim_bulan": (dim_bulan, 12),
    "dim_lokasi": (dim_lokasi, 35),
    "dim_jenis_kendaraan": (dim_jenis_kendaraan, 4),
    "dim_jenis_transportasi": (dim_jenis_transportasi, 2),
    "dim_jenis_jalan": (dim_jenis_jalan, 3),
    "dim_indikator_kecelakaan": (dim_indikator_kecelakaan, 5)
}

for nama, (df, jumlah) in daftar_dimensi.items():
    # Memeriksa jumlah baris
    assert len(df) == jumlah, f"Jumlah baris {nama} tidak sesuai"

    # Memeriksa missing values
    assert df.notna().all().all(), f"Ada missing values pada {nama}"

    # Memeriksa ID dimensi
    assert df.iloc[:, 0].is_unique, f"ID duplikat pada {nama}"

    print(f"{nama}: {len(df)} baris - OK")

print("\nValidasi seluruh dimensi berhasil!")

dim_tahun: 2 baris - OK
dim_bulan: 12 baris - OK
dim_lokasi: 35 baris - OK
dim_jenis_kendaraan: 4 baris - OK
dim_jenis_transportasi: 2 baris - OK
dim_jenis_jalan: 3 baris - OK
dim_indikator_kecelakaan: 5 baris - OK

Validasi seluruh dimensi berhasil!


# Membuat Fakta

In [121]:
# ======================
# Membuat Fakta Transportasi
# ======================

# Menyiapkan data penerbangan domestik
pesawat_fact = pesawat_clean[
    ["tahun", "bulan", "penumpang"]
].copy()

pesawat_fact["jenis_transportasi"] = "Penerbangan Domestik"

# Menyiapkan data angkutan laut
laut_fact = debarkasi_angkutan_laut[
    ["tahun", "bulan", "penumpang"]
].copy()

laut_fact["jenis_transportasi"] = "Angkutan Laut"

# Menggabungkan kedua data
transportasi = pd.concat(
    [laut_fact, pesawat_fact],
    ignore_index=True
)

# Mengubah tipe data
transportasi["tahun"] = transportasi["tahun"].astype(int)
transportasi["penumpang"] = pd.to_numeric(
    transportasi["penumpang"],
    errors="raise"
)

display(transportasi.head(10))

,tahun,bulan,penumpang,jenis_transportasi
0,2020,Januari,18821,Angkutan Laut
1,2020,Februari,14667,Angkutan Laut
2,2020,Maret,17103,Angkutan Laut
3,2020,April,5024,Angkutan Laut
4,2020,Mei,2267,Angkutan Laut
5,2020,Juni,6527,Angkutan Laut
6,2020,Juli,7977,Angkutan Laut
7,2020,Agustus,9174,Angkutan Laut
8,2020,September,9272,Angkutan Laut
9,2020,Oktober,13432,Angkutan Laut


In [122]:
# ======================
# Mapping Dimensi Fakta Transportasi
# ======================

# Menghubungkan dengan dimensi tahun
fakta_transportasi = transportasi.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi bulan
fakta_transportasi = fakta_transportasi.merge(
    dim_bulan[["id_bulan", "nama_bulan"]],
    left_on="bulan",
    right_on="nama_bulan",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi jenis transportasi
fakta_transportasi = fakta_transportasi.merge(
    dim_jenis_transportasi,
    left_on="jenis_transportasi",
    right_on="nama_jenis_transportasi",
    how="left",
    validate="many_to_one"
)

# Mengambil kolom sesuai struktur fact table
fakta_transportasi = fakta_transportasi[
    [
        "id_bulan",
        "id_tahun",
        "id_jenis_transportasi",
        "penumpang"
    ]
].copy()

# Mengurutkan data
fakta_transportasi = fakta_transportasi.sort_values(
    ["id_tahun", "id_jenis_transportasi", "id_bulan"]
).reset_index(drop=True)

display(fakta_transportasi)

,id_bulan,id_tahun,id_jenis_transportasi,penumpang
0,1,1,1,18821
1,2,1,1,14667
2,3,1,1,17103
3,4,1,1,5024
4,5,1,1,2267
5,6,1,1,6527
6,7,1,1,7977
7,8,1,1,9174
8,9,1,1,9272
9,10,1,1,13432


In [123]:
# ======================
# Validasi Fakta Transportasi
# ======================

print("Jumlah baris:", len(fakta_transportasi))

print("\nMissing values:")
print(fakta_transportasi.isna().sum())

print("\nJumlah data per tahun dan jenis transportasi:")
print(
    fakta_transportasi.groupby(
        ["id_tahun", "id_jenis_transportasi"]
    ).size()
)

# Memastikan jumlah data lengkap
assert len(fakta_transportasi) == 48

# Memastikan setiap kombinasi tahun dan moda memiliki 12 bulan
assert (
    fakta_transportasi.groupby(
        ["id_tahun", "id_jenis_transportasi"]
    )["id_bulan"].nunique() == 12
).all()

# Memastikan seluruh foreign key ditemukan
assert fakta_transportasi[
    ["id_bulan", "id_tahun", "id_jenis_transportasi"]
].notna().all().all()

# Memastikan tidak ada duplikasi
assert not fakta_transportasi.duplicated(
    subset=["id_bulan", "id_tahun", "id_jenis_transportasi"]
).any()

# Memastikan penumpang valid
assert fakta_transportasi["penumpang"].notna().all()
assert (fakta_transportasi["penumpang"] >= 0).all()

# Memastikan hasil angkutan laut sesuai BPS
total_laut = (
    fakta_transportasi[
        fakta_transportasi["id_jenis_transportasi"] == 1
    ]
    .merge(dim_tahun, on="id_tahun")
    .groupby("tahun")["penumpang"]
    .sum()
)

assert total_laut.loc[2020] == 133604
assert total_laut.loc[2021] == 152016

print("\nValidasi fakta transportasi berhasil!")

Jumlah baris: 48

Missing values:
id_bulan                 0
id_tahun                 0
id_jenis_transportasi    0
penumpang                0
dtype: int64

Jumlah data per tahun dan jenis transportasi:
id_tahun  id_jenis_transportasi
1         1                        12
          2                        12
2         1                        12
          2                        12
dtype: int64

Validasi fakta transportasi berhasil!


In [124]:
# ======================
# Membuat Fakta Kecelakaan
# ======================

# Mengubah data kecelakaan menjadi format long
laka_long = laka_lantas.melt(
    id_vars=["kabupaten_kota"],
    var_name="indikator_tahun",
    value_name="nilai"
)

# Memisahkan indikator dan tahun
laka_long["tahun"] = (
    laka_long["indikator_tahun"]
    .str.extract(r"_(2020|2021)$")[0]
    .astype(int)
)

laka_long["indikator"] = (
    laka_long["indikator_tahun"]
    .str.replace(r"_(2020|2021)$", "", regex=True)
)

# Menyamakan format nama wilayah
laka_long["kabupaten_kota"] = (
    laka_long["kabupaten_kota"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Mengubah nilai menjadi numerik
laka_long["nilai"] = pd.to_numeric(
    laka_long["nilai"],
    errors="raise"
)

# Memilih kolom yang diperlukan
laka_long = laka_long[
    ["kabupaten_kota", "tahun", "indikator", "nilai"]
]

display(laka_long.head(10))
print("Jumlah baris:", len(laka_long))

,kabupaten_kota,tahun,indikator,nilai
0,cilacap,2020,jumlah_kecelakaan,851.0
1,banyumas,2020,jumlah_kecelakaan,1574.0
2,purbalingga,2020,jumlah_kecelakaan,439.0
3,banjarnegara,2020,jumlah_kecelakaan,451.0
4,kebumen,2020,jumlah_kecelakaan,618.0
5,purworejo,2020,jumlah_kecelakaan,505.0
6,wonosobo,2020,jumlah_kecelakaan,276.0
7,magelang,2020,jumlah_kecelakaan,782.0
8,boyolali,2020,jumlah_kecelakaan,761.0
9,klaten,2020,jumlah_kecelakaan,1003.0


Jumlah baris: 350


In [125]:
# ======================
# Mapping Dimensi Fakta Kecelakaan
# ======================

# Menghubungkan dengan dimensi lokasi
fakta_kecelakaan = laka_long.merge(
    dim_lokasi,
    on="kabupaten_kota",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi tahun
fakta_kecelakaan = fakta_kecelakaan.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi indikator
fakta_kecelakaan = fakta_kecelakaan.merge(
    dim_indikator_kecelakaan,
    left_on="indikator",
    right_on="nama_indikator",
    how="left",
    validate="many_to_one"
)

# Memastikan seluruh foreign key ditemukan
assert fakta_kecelakaan[
    ["id_lokasi", "id_tahun", "id_indikator"]
].notna().all().all(), "Ada data yang gagal mapping dimensi"

# Memilih kolom sesuai struktur fact table
fakta_kecelakaan = fakta_kecelakaan[
    ["id_lokasi", "id_tahun", "id_indikator", "nilai"]
].copy()

# Mengubah tipe data foreign key menjadi integer
for col in ["id_lokasi", "id_tahun", "id_indikator"]:
    fakta_kecelakaan[col] = fakta_kecelakaan[col].astype(int)

# Mengurutkan data
fakta_kecelakaan = fakta_kecelakaan.sort_values(
    ["id_tahun", "id_indikator", "id_lokasi"]
).reset_index(drop=True)

display(fakta_kecelakaan)

,id_lokasi,id_tahun,id_indikator,nilai
0,1,1,1,851.0
1,2,1,1,1574.0
2,3,1,1,439.0
3,4,1,1,451.0
4,5,1,1,618.0
...,...,...,...,...
345,31,2,5,53.0
346,32,2,5,48.0
347,33,2,5,147.0
348,34,2,5,21.0


In [126]:
# ======================
# Validasi Fakta Kecelakaan
# ======================

print("Jumlah baris:", len(fakta_kecelakaan))

print("\nMissing values:")
print(fakta_kecelakaan.isna().sum())

print("\nJumlah data per tahun:")
print(
    fakta_kecelakaan.groupby("id_tahun").size()
)

print("\nJumlah data per indikator:")
print(
    fakta_kecelakaan.groupby("id_indikator").size()
)

# Validasi jumlah data
assert len(fakta_kecelakaan) == 350

# Validasi missing values
assert fakta_kecelakaan.notna().all().all()

# Validasi duplikasi kombinasi foreign key
assert not fakta_kecelakaan.duplicated(
    subset=["id_lokasi", "id_tahun", "id_indikator"]
).any()

# Validasi jumlah wilayah, tahun, dan indikator
assert fakta_kecelakaan["id_lokasi"].nunique() == 35
assert fakta_kecelakaan["id_tahun"].nunique() == 2
assert fakta_kecelakaan["id_indikator"].nunique() == 5

# Validasi setiap kombinasi tahun dan indikator
assert (
    fakta_kecelakaan.groupby(
        ["id_tahun", "id_indikator"]
    )["id_lokasi"].nunique() == 35
).all()

# Validasi nilai kecelakaan
assert (fakta_kecelakaan["nilai"] >= 0).all()

# Memastikan total nilai tidak berubah setelah mapping
assert np.isclose(
    laka_long["nilai"].sum(),
    fakta_kecelakaan["nilai"].sum()
)

# Memastikan kabupaten dan kota memiliki ID berbeda
for nama in ["magelang", "semarang", "pekalongan", "tegal"]:
    id_kabupaten = dim_lokasi.loc[
        dim_lokasi["kabupaten_kota"] == nama,
        "id_lokasi"
    ].iloc[0]

    id_kota = dim_lokasi.loc[
        dim_lokasi["kabupaten_kota"] == f"kota_{nama}",
        "id_lokasi"
    ].iloc[0]

    assert id_kabupaten != id_kota

    print(f"{nama}: Kabupaten ID {id_kabupaten}, Kota ID {id_kota}")

print("\nValidasi fakta kecelakaan berhasil!")

Jumlah baris: 350

Missing values:
id_lokasi       0
id_tahun        0
id_indikator    0
nilai           0
dtype: int64

Jumlah data per tahun:
id_tahun
1    175
2    175
dtype: int64

Jumlah data per indikator:
id_indikator
1    70
2    70
3    70
4    70
5    70
dtype: int64
magelang: Kabupaten ID 8, Kota ID 30
semarang: Kabupaten ID 22, Kota ID 33
pekalongan: Kabupaten ID 26, Kota ID 34
tegal: Kabupaten ID 28, Kota ID 35

Validasi fakta kecelakaan berhasil!


In [127]:
# ======================
# Membuat Fakta Kepadatan
# ======================

# Mengambil jenis kendaraan dari dimensi
kolom_kendaraan = dim_jenis_kendaraan[
    "nama_jenis_kendaraan"
].tolist()

# Mengubah data kendaraan dari wide ke long
kendaraan_long = kendaraan_bermotor.melt(
    id_vars=["kabupaten_kota", "tahun"],
    value_vars=kolom_kendaraan,
    var_name="jenis_kendaraan",
    value_name="total_kendaraan"
)

# Mengambil data panjang jalan
jalan_fact = jalan_clean[
    ["kabupaten_kota", "tahun", "total_panjang_jalan"]
].copy()

# Menyamakan format nama wilayah
for df in [kendaraan_long, jalan_fact]:
    df["kabupaten_kota"] = (
        df["kabupaten_kota"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    df["tahun"] = df["tahun"].astype(int)

# Memastikan data panjang jalan tidak duplikat
assert not jalan_fact.duplicated(
    subset=["kabupaten_kota", "tahun"]
).any()

# Menggabungkan kendaraan dan panjang jalan
kepadatan = kendaraan_long.merge(
    jalan_fact,
    on=["kabupaten_kota", "tahun"],
    how="left",
    validate="many_to_one"
)

display(kepadatan.head(10))
print("Jumlah baris:", len(kepadatan))

,kabupaten_kota,tahun,jenis_kendaraan,total_kendaraan,total_panjang_jalan
0,cilacap,2020,mobil_penumpang,44767.0,1526
1,banyumas,2020,mobil_penumpang,60915.0,1719
2,purbalingga,2020,mobil_penumpang,23347.0,922
3,banjarnegara,2020,mobil_penumpang,20088.0,1086
4,kebumen,2020,mobil_penumpang,25038.0,1046
5,purworejo,2020,mobil_penumpang,21659.0,909
6,wonosobo,2020,mobil_penumpang,20135.0,1141
7,magelang,2020,mobil_penumpang,43327.0,1157
8,boyolali,2020,mobil_penumpang,38230.0,763
9,klaten,2020,mobil_penumpang,55235.0,838


Jumlah baris: 280


In [128]:
# ======================
# Mapping Dimensi Fakta Kepadatan
# ======================

# Menghubungkan dengan dimensi lokasi
fakta_kepadatan = kepadatan.merge(
    dim_lokasi,
    on="kabupaten_kota",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi tahun
fakta_kepadatan = fakta_kepadatan.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi jenis kendaraan
fakta_kepadatan = fakta_kepadatan.merge(
    dim_jenis_kendaraan,
    left_on="jenis_kendaraan",
    right_on="nama_jenis_kendaraan",
    how="left",
    validate="many_to_one"
)

# Memastikan seluruh mapping berhasil
assert fakta_kepadatan[
    [
        "id_lokasi",
        "id_tahun",
        "id_jenis_kendaraan",
        "total_panjang_jalan"
    ]
].notna().all().all()

# Memastikan panjang jalan valid
assert (fakta_kepadatan["total_panjang_jalan"] > 0).all()

# Menghitung kendaraan per kilometer
fakta_kepadatan["kendaraan_per_km"] = (
    fakta_kepadatan["total_kendaraan"]
    / fakta_kepadatan["total_panjang_jalan"]
)

# Memilih kolom fact table
fakta_kepadatan = fakta_kepadatan[
    [
        "id_lokasi",
        "id_tahun",
        "id_jenis_kendaraan",
        "total_kendaraan",
        "total_panjang_jalan",
        "kendaraan_per_km"
    ]
].copy()

# Mengubah foreign key menjadi integer
for col in ["id_lokasi", "id_tahun", "id_jenis_kendaraan"]:
    fakta_kepadatan[col] = fakta_kepadatan[col].astype(int)

# Mengurutkan data
fakta_kepadatan = fakta_kepadatan.sort_values(
    ["id_tahun", "id_lokasi", "id_jenis_kendaraan"]
).reset_index(drop=True)

display(fakta_kepadatan)

,id_lokasi,id_tahun,id_jenis_kendaraan,total_kendaraan,total_panjang_jalan,kendaraan_per_km
0,1,1,1,44767.0,1526,29.336173
1,1,1,2,793.0,1526,0.519659
2,1,1,3,20762.0,1526,13.605505
3,1,1,4,752008.0,1526,492.796855
4,2,1,1,60915.0,1719,35.436300
...,...,...,...,...,...,...
275,34,2,4,201732.0,153,1318.509804
276,35,2,1,13703.0,244,56.159836
277,35,2,2,801.0,244,3.282787
278,35,2,3,5942.0,244,24.352459


In [129]:
# ======================
# Validasi Fakta Kepadatan
# ======================

print("Jumlah baris:", len(fakta_kepadatan))

print("\nMissing values:")
print(fakta_kepadatan.isna().sum())

print("\nJumlah data per tahun:")
print(fakta_kepadatan.groupby("id_tahun").size())

print("\nJumlah data per jenis kendaraan:")
print(fakta_kepadatan.groupby("id_jenis_kendaraan").size())

# Validasi jumlah baris
assert len(fakta_kepadatan) == 280

# Validasi missing values
assert fakta_kepadatan.notna().all().all()

# Validasi duplikasi
assert not fakta_kepadatan.duplicated(
    subset=["id_lokasi", "id_tahun", "id_jenis_kendaraan"]
).any()

# Validasi setiap wilayah memiliki 4 jenis kendaraan per tahun
assert (
    fakta_kepadatan.groupby(
        ["id_lokasi", "id_tahun"]
    )["id_jenis_kendaraan"].nunique() == 4
).all()

# Validasi nilai
assert (fakta_kepadatan["total_kendaraan"] >= 0).all()
assert (fakta_kepadatan["total_panjang_jalan"] > 0).all()
assert np.isfinite(fakta_kepadatan["kendaraan_per_km"]).all()

# Memastikan jumlah kendaraan tidak berubah
assert np.isclose(
    kendaraan_long["total_kendaraan"].sum(),
    fakta_kepadatan["total_kendaraan"].sum()
)

# Memastikan perhitungan rasio benar
assert np.allclose(
    fakta_kepadatan["kendaraan_per_km"],
    fakta_kepadatan["total_kendaraan"]
    / fakta_kepadatan["total_panjang_jalan"]
)

print("\nValidasi fakta kepadatan berhasil!")

Jumlah baris: 280

Missing values:
id_lokasi              0
id_tahun               0
id_jenis_kendaraan     0
total_kendaraan        0
total_panjang_jalan    0
kendaraan_per_km       0
dtype: int64

Jumlah data per tahun:
id_tahun
1    140
2    140
dtype: int64

Jumlah data per jenis kendaraan:
id_jenis_kendaraan
1    70
2    70
3    70
4    70
dtype: int64

Validasi fakta kepadatan berhasil!


In [130]:
# ======================
# Membuat Fakta Risiko
# ======================

# Mengambil ID indikator jumlah kecelakaan
id_kecelakaan = dim_indikator_kecelakaan.loc[
    dim_indikator_kecelakaan["nama_indikator"] == "jumlah_kecelakaan",
    "id_indikator"
].iloc[0]

# Mengambil jumlah kecelakaan dari fakta_kecelakaan
risiko_laka = fakta_kecelakaan.loc[
    fakta_kecelakaan["id_indikator"] == id_kecelakaan,
    ["id_lokasi", "id_tahun", "nilai"]
].copy()

# Mengubah nama kolom
risiko_laka = risiko_laka.rename(
    columns={"nilai": "jumlah_kecelakaan"}
)

# Mengambil total panjang jalan
risiko_jalan = jalan_clean[
    ["kabupaten_kota", "tahun", "total_panjang_jalan"]
].copy()

# Menyamakan format nama wilayah
risiko_jalan["kabupaten_kota"] = (
    risiko_jalan["kabupaten_kota"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Menghubungkan panjang jalan dengan dimensi lokasi
risiko_jalan = risiko_jalan.merge(
    dim_lokasi,
    on="kabupaten_kota",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi tahun
risiko_jalan = risiko_jalan.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Memastikan seluruh wilayah dan tahun berhasil dipetakan
assert risiko_jalan[
    ["id_lokasi", "id_tahun"]
].notna().all().all()

# Menggabungkan data kecelakaan dan panjang jalan
risiko = risiko_laka.merge(
    risiko_jalan[
        ["id_lokasi", "id_tahun", "total_panjang_jalan"]
    ],
    on=["id_lokasi", "id_tahun"],
    how="left",
    validate="one_to_one"
)

display(risiko.head(10))
print("Jumlah baris:", len(risiko))

,id_lokasi,id_tahun,jumlah_kecelakaan,total_panjang_jalan
0,1,1,851.0,1526
1,2,1,1574.0,1719
2,3,1,439.0,922
3,4,1,451.0,1086
4,5,1,618.0,1046
5,6,1,505.0,909
6,7,1,276.0,1141
7,8,1,782.0,1157
8,9,1,761.0,763
9,10,1,1003.0,838


Jumlah baris: 70


In [131]:
# ======================
# Menghitung Fakta Risiko
# ======================

# Memastikan data panjang jalan tersedia dan valid
assert risiko["total_panjang_jalan"].notna().all()
assert (risiko["total_panjang_jalan"] > 0).all()

# Menghitung rasio kecelakaan per kilometer
risiko["rasio_kecelakaan_per_km"] = (
    risiko["jumlah_kecelakaan"]
    / risiko["total_panjang_jalan"]
)

# Membuat tabel fakta risiko
fakta_risiko = risiko[
    [
        "id_lokasi",
        "id_tahun",
        "jumlah_kecelakaan",
        "total_panjang_jalan",
        "rasio_kecelakaan_per_km"
    ]
].copy()

# Mengubah foreign key menjadi integer
for col in ["id_lokasi", "id_tahun"]:
    fakta_risiko[col] = fakta_risiko[col].astype(int)

# Mengurutkan data
fakta_risiko = fakta_risiko.sort_values(
    ["id_tahun", "id_lokasi"]
).reset_index(drop=True)

display(fakta_risiko)

,id_lokasi,id_tahun,jumlah_kecelakaan,total_panjang_jalan,rasio_kecelakaan_per_km
0,1,1,851.0,1526,0.557667
1,2,1,1574.0,1719,0.915649
2,3,1,439.0,922,0.476139
3,4,1,451.0,1086,0.415285
4,5,1,618.0,1046,0.590822
...,...,...,...,...,...
65,31,2,898.0,238,3.773109
66,32,2,246.0,350,0.702857
67,33,2,906.0,948,0.955696
68,34,2,103.0,153,0.673203


In [132]:
# ======================
# Validasi Fakta Risiko
# ======================

print("Jumlah baris:", len(fakta_risiko))

print("\nMissing values:")
print(fakta_risiko.isna().sum())

print("\nJumlah data per tahun:")
print(fakta_risiko.groupby("id_tahun").size())

print("\nJumlah wilayah:")
print(fakta_risiko["id_lokasi"].nunique())

# Validasi jumlah baris
assert len(fakta_risiko) == 70

# Validasi missing values
assert fakta_risiko.notna().all().all()

# Validasi duplikasi
assert not fakta_risiko.duplicated(
    subset=["id_lokasi", "id_tahun"]
).any()

# Memastikan tiap tahun memiliki 35 wilayah
assert (
    fakta_risiko.groupby("id_tahun")["id_lokasi"].nunique()
    == 35
).all()

# Validasi nilai
assert (fakta_risiko["jumlah_kecelakaan"] >= 0).all()
assert (fakta_risiko["total_panjang_jalan"] > 0).all()

# Validasi hasil perhitungan
assert np.isfinite(
    fakta_risiko["rasio_kecelakaan_per_km"]
).all()

assert np.allclose(
    fakta_risiko["rasio_kecelakaan_per_km"],
    fakta_risiko["jumlah_kecelakaan"]
    / fakta_risiko["total_panjang_jalan"]
)

# Memastikan jumlah kecelakaan tidak berubah
assert np.isclose(
    risiko_laka["jumlah_kecelakaan"].sum(),
    fakta_risiko["jumlah_kecelakaan"].sum()
)

print("\nValidasi fakta risiko berhasil!")

Jumlah baris: 70

Missing values:
id_lokasi                  0
id_tahun                   0
jumlah_kecelakaan          0
total_panjang_jalan        0
rasio_kecelakaan_per_km    0
dtype: int64

Jumlah data per tahun:
id_tahun
1    35
2    35
dtype: int64

Jumlah wilayah:
35

Validasi fakta risiko berhasil!


# Validasi Akhir Hasil ETL

In [133]:
# ======================
# Validasi Struktur Seluruh Tabel
# ======================

# Daftar tabel dimensi dan jumlah baris yang diharapkan
tabel_dimensi = {
    "dim_tahun": (dim_tahun, 2, "id_tahun"),
    "dim_bulan": (dim_bulan, 12, "id_bulan"),
    "dim_lokasi": (dim_lokasi, 35, "id_lokasi"),
    "dim_jenis_kendaraan": (dim_jenis_kendaraan, 4, "id_jenis_kendaraan"),
    "dim_jenis_transportasi": (dim_jenis_transportasi, 2, "id_jenis_transportasi"),
    "dim_jenis_jalan": (dim_jenis_jalan, 3, "id_jenis_jalan"),
    "dim_indikator_kecelakaan": (dim_indikator_kecelakaan, 5, "id_indikator")
}

# Daftar tabel fakta dan jumlah baris yang diharapkan
tabel_fakta = {
    "fakta_transportasi": (fakta_transportasi, 48),
    "fakta_kecelakaan": (fakta_kecelakaan, 350),
    "fakta_kepadatan": (fakta_kepadatan, 280),
    "fakta_risiko": (fakta_risiko, 70)
}

# Validasi tabel dimensi
for nama, (df, jumlah, id_col) in tabel_dimensi.items():
    assert len(df) == jumlah, f"Jumlah baris {nama} tidak sesuai"
    assert df.notna().all().all(), f"Missing values pada {nama}"
    assert df[id_col].is_unique, f"ID duplikat pada {nama}"

    print(f"{nama}: {len(df)} baris - OK")

print()

# Validasi tabel fakta
for nama, (df, jumlah) in tabel_fakta.items():
    assert len(df) == jumlah, f"Jumlah baris {nama} tidak sesuai"
    assert df.notna().all().all(), f"Missing values pada {nama}"

    print(f"{nama}: {len(df)} baris - OK")

# Menghitung total baris
total_dimensi = sum(len(df) for df, _, _ in tabel_dimensi.values())
total_fakta = sum(len(df) for df, _ in tabel_fakta.values())

print("\nTotal baris dimensi:", total_dimensi)
print("Total baris fakta:", total_fakta)
print("Total seluruh baris:", total_dimensi + total_fakta)

assert total_dimensi + total_fakta == 811

print("\nValidasi struktur seluruh tabel berhasil!")

dim_tahun: 2 baris - OK
dim_bulan: 12 baris - OK
dim_lokasi: 35 baris - OK
dim_jenis_kendaraan: 4 baris - OK
dim_jenis_transportasi: 2 baris - OK
dim_jenis_jalan: 3 baris - OK
dim_indikator_kecelakaan: 5 baris - OK

fakta_transportasi: 48 baris - OK
fakta_kecelakaan: 350 baris - OK
fakta_kepadatan: 280 baris - OK
fakta_risiko: 70 baris - OK

Total baris dimensi: 63
Total baris fakta: 748
Total seluruh baris: 811

Validasi struktur seluruh tabel berhasil!


In [134]:
# ======================
# Validasi Foreign Key
# ======================

# Daftar hubungan antara tabel fakta dan dimensi
relasi_fk = [
    (fakta_transportasi, "id_bulan", dim_bulan),
    (fakta_transportasi, "id_tahun", dim_tahun),
    (fakta_transportasi, "id_jenis_transportasi", dim_jenis_transportasi),

    (fakta_kecelakaan, "id_lokasi", dim_lokasi),
    (fakta_kecelakaan, "id_tahun", dim_tahun),
    (fakta_kecelakaan, "id_indikator", dim_indikator_kecelakaan),

    (fakta_kepadatan, "id_lokasi", dim_lokasi),
    (fakta_kepadatan, "id_tahun", dim_tahun),
    (fakta_kepadatan, "id_jenis_kendaraan", dim_jenis_kendaraan),

    (fakta_risiko, "id_lokasi", dim_lokasi),
    (fakta_risiko, "id_tahun", dim_tahun)
]

# Memeriksa seluruh foreign key
for df_fakta, kolom_fk, df_dimensi in relasi_fk:

    id_valid = set(df_dimensi[kolom_fk])

    id_fakta = set(df_fakta[kolom_fk])

    id_tidak_valid = id_fakta - id_valid

    assert len(id_tidak_valid) == 0, (
        f"Foreign key {kolom_fk} tidak valid: {id_tidak_valid}"
    )

    print(f"{kolom_fk} - OK")

print("\nSeluruh foreign key valid!")

id_bulan - OK
id_tahun - OK
id_jenis_transportasi - OK
id_lokasi - OK
id_tahun - OK
id_indikator - OK
id_lokasi - OK
id_tahun - OK
id_jenis_kendaraan - OK
id_lokasi - OK
id_tahun - OK

Seluruh foreign key valid!


In [135]:
# ======================
# Validasi Konsistensi Antar-Fakta
# ======================

# 1. Memeriksa panjang jalan pada fakta kepadatan

cek_panjang = fakta_kepadatan.groupby(
    ["id_lokasi", "id_tahun"]
).agg(
    panjang_jalan=("total_panjang_jalan", "first"),
    jumlah_nilai_panjang=("total_panjang_jalan", "nunique")
).reset_index()

# Setiap wilayah-tahun harus memakai satu nilai panjang jalan
assert (cek_panjang["jumlah_nilai_panjang"] == 1).all()

# Membandingkan dengan fakta risiko
cek_panjang = cek_panjang.merge(
    fakta_risiko[
        ["id_lokasi", "id_tahun", "total_panjang_jalan"]
    ],
    on=["id_lokasi", "id_tahun"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert len(cek_panjang) == 70
assert (cek_panjang["_merge"] == "both").all()

assert np.allclose(
    cek_panjang["panjang_jalan"],
    cek_panjang["total_panjang_jalan"]
)

print("Konsistensi panjang jalan - OK")


# 2. Memeriksa jumlah kecelakaan pada fakta risiko

id_kecelakaan = dim_indikator_kecelakaan.loc[
    dim_indikator_kecelakaan["nama_indikator"] == "jumlah_kecelakaan",
    "id_indikator"
].iloc[0]

cek_kecelakaan = fakta_kecelakaan.loc[
    fakta_kecelakaan["id_indikator"] == id_kecelakaan,
    ["id_lokasi", "id_tahun", "nilai"]
].copy()

cek_kecelakaan = cek_kecelakaan.rename(
    columns={"nilai": "jumlah_kecelakaan_asli"}
)

cek_kecelakaan = cek_kecelakaan.merge(
    fakta_risiko[
        ["id_lokasi", "id_tahun", "jumlah_kecelakaan"]
    ],
    on=["id_lokasi", "id_tahun"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert len(cek_kecelakaan) == 70
assert (cek_kecelakaan["_merge"] == "both").all()

assert np.allclose(
    cek_kecelakaan["jumlah_kecelakaan_asli"],
    cek_kecelakaan["jumlah_kecelakaan"]
)

print("Konsistensi jumlah kecelakaan - OK")


# 3. Memeriksa kelengkapan fakta transportasi

cek_transportasi = fakta_transportasi.groupby(
    ["id_tahun", "id_jenis_transportasi"]
).agg(
    jumlah_baris=("id_bulan", "size"),
    jumlah_bulan=("id_bulan", "nunique")
).reset_index()

assert len(cek_transportasi) == 4
assert (cek_transportasi["jumlah_baris"] == 12).all()
assert (cek_transportasi["jumlah_bulan"] == 12).all()

print("Kelengkapan transportasi - OK")

print("\nValidasi konsistensi antar-fakta berhasil!")

Konsistensi panjang jalan - OK
Konsistensi jumlah kecelakaan - OK
Kelengkapan transportasi - OK

Validasi konsistensi antar-fakta berhasil!


# Connect ke MySQL

In [137]:
# ======================
# Koneksi ke Database MySQL
# ======================

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL
from getpass import getpass

# Konfigurasi database
host = "127.0.0.1"
port = 3306
user = "root"
password = getpass("Password MySQL (Enter jika kosong): ")
database = "tugas_iak"

# Membuat URL koneksi
db_url = URL.create(
    drivername="mysql+pymysql",
    username=user,
    password=password,
    host=host,
    port=port,
    database=database
)

# Membuat engine
engine = create_engine(
    db_url,
    pool_pre_ping=True
)

# Memeriksa koneksi
with engine.connect() as conn:
    nama_database = conn.execute(
        text("SELECT DATABASE()")
    ).scalar()

    versi_mysql = conn.execute(
        text("SELECT VERSION()")
    ).scalar()

print("Koneksi MySQL berhasil!")
print("Database:", nama_database)
print("Versi MySQL:", versi_mysql)

assert nama_database == "tugas_iak"

Koneksi MySQL berhasil!
Database: tugas_iak
Versi MySQL: 8.0.30


# Loading ke Mysql

In [138]:
# ======================
# Menyiapkan Data untuk Loading
# ======================

# Daftar tabel dan dataframe yang akan dimasukkan
data_load = {
    "dim_tahun": dim_tahun,
    "dim_bulan": dim_bulan,
    "dim_lokasi": dim_lokasi,
    "dim_jenis_kendaraan": dim_jenis_kendaraan,
    "dim_jenis_transportasi": dim_jenis_transportasi,
    "dim_jenis_jalan": dim_jenis_jalan,
    "dim_indikator_kecelakaan": dim_indikator_kecelakaan,

    "fakta_transportasi": fakta_transportasi,
    "fakta_kecelakaan": fakta_kecelakaan,
    "fakta_kepadatan": fakta_kepadatan,
    "fakta_risiko": fakta_risiko
}

# Kolom yang akan dimasukkan ke masing-masing tabel
kolom_load = {
    "dim_tahun": ["id_tahun", "tahun"],
    "dim_bulan": ["id_bulan", "nama_bulan", "kuartal"],
    "dim_lokasi": ["id_lokasi", "kabupaten_kota"],
    "dim_jenis_kendaraan": [
        "id_jenis_kendaraan", "nama_jenis_kendaraan"
    ],
    "dim_jenis_transportasi": [
        "id_jenis_transportasi", "nama_jenis_transportasi"
    ],
    "dim_jenis_jalan": [
        "id_jenis_jalan", "nama_jenis_jalan"
    ],
    "dim_indikator_kecelakaan": [
        "id_indikator", "nama_indikator"
    ],

    "fakta_transportasi": [
        "id_bulan", "id_tahun",
        "id_jenis_transportasi", "penumpang"
    ],
    "fakta_kecelakaan": [
        "id_lokasi", "id_tahun",
        "id_indikator", "nilai"
    ],
    "fakta_kepadatan": [
        "id_lokasi", "id_tahun",
        "id_jenis_kendaraan", "total_kendaraan",
        "total_panjang_jalan", "kendaraan_per_km"
    ],
    "fakta_risiko": [
        "id_lokasi", "id_tahun",
        "jumlah_kecelakaan",
        "total_panjang_jalan",
        "rasio_kecelakaan_per_km"
    ]
}

# Validasi kesiapan data
for nama, df in data_load.items():

    assert set(df.columns) == set(kolom_load[nama]), (
        f"Kolom {nama} tidak sesuai"
    )

    assert df[kolom_load[nama]].notna().all().all(), (
        f"Ada missing values pada {nama}"
    )

    print(f"{nama}: {len(df)} baris siap dimuat")

print("\nSeluruh data siap untuk loading!")

dim_tahun: 2 baris siap dimuat
dim_bulan: 12 baris siap dimuat
dim_lokasi: 35 baris siap dimuat
dim_jenis_kendaraan: 4 baris siap dimuat
dim_jenis_transportasi: 2 baris siap dimuat
dim_jenis_jalan: 3 baris siap dimuat
dim_indikator_kecelakaan: 5 baris siap dimuat
fakta_transportasi: 48 baris siap dimuat
fakta_kecelakaan: 350 baris siap dimuat
fakta_kepadatan: 280 baris siap dimuat
fakta_risiko: 70 baris siap dimuat

Seluruh data siap untuk loading!


In [140]:
# ======================
# Loading Data ke MySQL
# ======================

# Memastikan seluruh tabel sudah ada
tabel_mysql = set(inspect(engine).get_table_names())

assert set(data_load).issubset(tabel_mysql), (
    "Ada tabel data warehouse yang belum tersedia di MySQL"
)

# Memulai transaksi loading
with engine.begin() as conn:

    # Memastikan database yang digunakan benar
    assert conn.execute(
        text("SELECT DATABASE()")
    ).scalar() == "tugas_iak"

    # Memastikan seluruh tabel masih kosong
    for nama in data_load:

        jumlah_awal = conn.execute(
            text(f"SELECT COUNT(*) FROM `{nama}`")
        ).scalar()

        assert jumlah_awal == 0, (
            f"Tabel {nama} belum kosong. "
            "Jangan loading dua kali tanpa reset."
        )

    # Memasukkan data dimensi terlebih dahulu,
    # kemudian data fakta
    for nama, df in data_load.items():

        df[kolom_load[nama]].to_sql(
            name=nama,
            con=conn,
            if_exists="append",
            index=False,
            chunksize=500,
            method="multi"
        )

        # Memeriksa jumlah baris sebelum commit
        jumlah_mysql = conn.execute(
            text(f"SELECT COUNT(*) FROM `{nama}`")
        ).scalar()

        assert jumlah_mysql == len(df), (
            f"Jumlah baris {nama} tidak sesuai"
        )

        print(f"{nama}: {jumlah_mysql} baris berhasil dimuat")

print("\nLoading seluruh data warehouse berhasil!")

dim_tahun: 2 baris berhasil dimuat
dim_bulan: 12 baris berhasil dimuat
dim_lokasi: 35 baris berhasil dimuat
dim_jenis_kendaraan: 4 baris berhasil dimuat
dim_jenis_transportasi: 2 baris berhasil dimuat
dim_jenis_jalan: 3 baris berhasil dimuat
dim_indikator_kecelakaan: 5 baris berhasil dimuat
fakta_transportasi: 48 baris berhasil dimuat
fakta_kecelakaan: 350 baris berhasil dimuat
fakta_kepadatan: 280 baris berhasil dimuat
fakta_risiko: 70 baris berhasil dimuat

Loading seluruh data warehouse berhasil!


In [141]:
# ======================
# Verifikasi Jumlah Baris MySQL
# ======================

hasil_verifikasi = []

with engine.connect() as conn:

    for nama, df in data_load.items():

        jumlah_mysql = conn.execute(
            text(f"SELECT COUNT(*) FROM `{nama}`")
        ).scalar()

        hasil_verifikasi.append({
            "nama_tabel": nama,
            "baris_python": len(df),
            "baris_mysql": jumlah_mysql,
            "status": (
                "Sesuai"
                if len(df) == jumlah_mysql
                else "Tidak Sesuai"
            )
        })

df_verifikasi = pd.DataFrame(hasil_verifikasi)

display(df_verifikasi)

assert (
    df_verifikasi["baris_python"]
    == df_verifikasi["baris_mysql"]
).all()

print("\nTotal baris MySQL:",
      df_verifikasi["baris_mysql"].sum())

assert df_verifikasi["baris_mysql"].sum() == 811

print("\nVerifikasi jumlah baris berhasil!")

,nama_tabel,baris_python,baris_mysql,status
0,dim_tahun,2,2,Sesuai
1,dim_bulan,12,12,Sesuai
2,dim_lokasi,35,35,Sesuai
3,dim_jenis_kendaraan,4,4,Sesuai
4,dim_jenis_transportasi,2,2,Sesuai
5,dim_jenis_jalan,3,3,Sesuai
6,dim_indikator_kecelakaan,5,5,Sesuai
7,fakta_transportasi,48,48,Sesuai
8,fakta_kecelakaan,350,350,Sesuai
9,fakta_kepadatan,280,280,Sesuai



Total baris MySQL: 811

Verifikasi jumlah baris berhasil!


In [142]:
# ======================
# Verifikasi Isi Data MySQL
# ======================

# Kolom kunci untuk mengurutkan data
kunci_tabel = {
    "dim_tahun": ["id_tahun"],
    "dim_bulan": ["id_bulan"],
    "dim_lokasi": ["id_lokasi"],
    "dim_jenis_kendaraan": ["id_jenis_kendaraan"],
    "dim_jenis_transportasi": ["id_jenis_transportasi"],
    "dim_jenis_jalan": ["id_jenis_jalan"],
    "dim_indikator_kecelakaan": ["id_indikator"],

    "fakta_transportasi": [
        "id_tahun", "id_jenis_transportasi", "id_bulan"
    ],
    "fakta_kecelakaan": [
        "id_tahun", "id_indikator", "id_lokasi"
    ],
    "fakta_kepadatan": [
        "id_tahun", "id_lokasi", "id_jenis_kendaraan"
    ],
    "fakta_risiko": [
        "id_tahun", "id_lokasi"
    ]
}

# Kolom FLOAT dan jumlah angka desimal untuk pembacaan SQL
# 0 untuk jumlah bilangan bulat, 8 untuk nilai desimal
kolom_float = {
    "fakta_transportasi": {
        "penumpang": 0
    },
    "fakta_kecelakaan": {
        "nilai": 8
    },
    "fakta_kepadatan": {
        "total_kendaraan": 0,
        "total_panjang_jalan": 8,
        "kendaraan_per_km": 8
    },
    "fakta_risiko": {
        "jumlah_kecelakaan": 0,
        "total_panjang_jalan": 8,
        "rasio_kecelakaan_per_km": 8
    }
}

# Membandingkan isi data Python dan MySQL
with engine.connect() as conn:

    for nama, df in data_load.items():

        # Menyiapkan query SQL
        kolom_sql = []

        for col in kolom_load[nama]:

            if col in kolom_float.get(nama, {}):
                desimal = kolom_float[nama][col]

                kolom_sql.append(
                    f"CAST(`{col}` AS DECIMAL(25,{desimal})) AS `{col}`"
                )
            else:
                kolom_sql.append(f"`{col}`")

        kolom_sql = ", ".join(kolom_sql)

        # Membaca data dari MySQL
        df_mysql = pd.read_sql_query(
            text(f"SELECT {kolom_sql} FROM `{nama}`"),
            conn
        )

        # Mengubah hasil CAST menjadi numerik
        for col in kolom_float.get(nama, {}):
            df_mysql[col] = pd.to_numeric(df_mysql[col])

        # Mengurutkan data Python
        df_python = (
            df[kolom_load[nama]]
            .sort_values(kunci_tabel[nama])
            .reset_index(drop=True)
        )

        # Mengurutkan data MySQL
        df_mysql = (
            df_mysql[kolom_load[nama]]
            .sort_values(kunci_tabel[nama])
            .reset_index(drop=True)
        )

        # Memastikan jumlah baris sesuai
        assert len(df_python) == len(df_mysql), (
            f"Jumlah baris {nama} tidak sesuai"
        )

        # Memastikan nilai bilangan bulat sama persis
        for col, desimal in kolom_float.get(nama, {}).items():
            if desimal == 0:
                assert np.array_equal(
                    df_python[col].to_numpy(),
                    df_mysql[col].to_numpy()
                ), f"Nilai {col} pada {nama} tidak sesuai"

        # Membandingkan isi data
        pd.testing.assert_frame_equal(
            df_python,
            df_mysql,
            check_dtype=False,
            check_exact=False,
            rtol=1e-6,
            atol=1e-4
        )

        print(f"{nama}: {len(df_mysql)} baris - OK")

print("\nVerifikasi seluruh isi data MySQL berhasil!")

dim_tahun: 2 baris - OK
dim_bulan: 12 baris - OK
dim_lokasi: 35 baris - OK
dim_jenis_kendaraan: 4 baris - OK
dim_jenis_transportasi: 2 baris - OK
dim_jenis_jalan: 3 baris - OK
dim_indikator_kecelakaan: 5 baris - OK
fakta_transportasi: 48 baris - OK
fakta_kecelakaan: 350 baris - OK
fakta_kepadatan: 280 baris - OK
fakta_risiko: 70 baris - OK

Verifikasi seluruh isi data MySQL berhasil!


In [88]:
# ======================
# Memeriksa Satuan Kerugian Materiil
# ======================

# Membaca data kecelakaan asli
laka_asli = pd.read_excel(
    file_laka,
    sheet_name=0,
    header=None
)

# Memeriksa header dan angka sumber
print("Header sumber:")
display(laka_asli.iloc[:8, 19:23])

print("\nNilai kerugian Cilacap dari sumber:")
print(laka_asli.iloc[7, 21])

print("\nNilai kerugian Cilacap setelah cleaning:")
display(
    laka_lantas.loc[
        laka_lantas["kabupaten_kota"]
        .astype(str).str.strip().str.lower() == "cilacap",
        ["kerugian_2020", "kerugian_2021"]
    ]
)

Header sumber:


,19,20,21,22
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,Kerugian Material (Ribu Rupiah),NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,Loss (Thousand Rupiah),NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,2018,2019.0,2020.0,2021.0
7,709300000,874150000.0,620700000.0,714600000.0



Nilai kerugian Cilacap dari sumber:
620700000.0

Nilai kerugian Cilacap setelah cleaning:


,kerugian_2020,kerugian_2021
0,620700.0,714600.0
